# 📊 Local Test 002: Pearson Correlation Selection Scan

This notebook tests our fast, vectorized Pearson Correlation selection strategy.
It calculates the high-frequency correlation between Z-Score daily aggregated features and sleep targets to identify stable feature thresholds.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

project_root = Path(os.getcwd()).resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.data.data_loader import DataLoader
from src.features.transformers import LeakFreeSubjectZScaler

print("✅ Correlation scan dependencies loaded successfully!")

In [ ]:
data_loader = DataLoader(data_dir="data")
metrics_df = data_loader.load_metrics()
processed_train_path = Path("data/processed/df_train_raw.parquet")

if processed_train_path.exists():
    df = pd.read_parquet(processed_train_path)
else:
    sensor_dict = data_loader.load_sensor_data()
    from src.data.make_dataset import build_training_dataset
    df = build_training_dataset(metrics_df, sensor_dict)

df = df.sort_values(by=['subject_id', 'date']).reset_index(drop=True)
print(f"Loaded dataset for correlation check: {df.shape}")

In [ ]:
print("🔄 Running high-performance vectorized correlation scanner...")
exclude_cols = {'subject_id', 'sleep_date', 'lifelog_date', 'date', 'timestamp', 'Q1','Q2','Q3','S1','S2','S3','S4'}
candidate_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in exclude_cols]

scaler = LeakFreeSubjectZScaler(id_col='subject_id', numeric_cols=candidate_cols)
df_scaled = scaler.fit_transform(df)

zscore_cols = [f"{col}_subject_zscore" for col in candidate_cols]
y_tgt = df['S1'].fillna(0.5).values
X_scaled = df_scaled[zscore_cols].fillna(0.0).values

# Vectorized Pearson correlation
X_centered = X_scaled - X_scaled.mean(axis=0)
y_centered = y_tgt - y_tgt.mean()

num = np.sum(X_centered * y_centered[:, np.newaxis], axis=0)
den = np.sqrt(np.sum(X_centered**2, axis=0)) * np.sqrt(np.sum(y_centered**2))
corrs = num / (den + 1e-8)

stability_scores = {col: abs(corrs[idx]) for idx, col in enumerate(zscore_cols)}
top_10 = sorted(stability_scores.items(), key=lambda x: x[1], reverse=True)[:10]

print("\n🏆 TOP 10 Highly Correlated Z-Score Features with S1 (Sleep Depth):")
for rank, (feat, score) in enumerate(top_10):
    print(f"  {rank+1}. {feat}: corr={score:.5f}")